# 02 - Genetic Algorithm Experiments

Este notebook consolida os experimentos da TASK-1.6 e compara baseline vs modelos otimizados por AG.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
summary_path = ROOT / 'results' / 'ga_summary.json'
experiments_path = ROOT / 'results' / 'experiments.json'

if not summary_path.exists():
    raise FileNotFoundError('ga_summary.json nao encontrado. Execute os experimentos primeiro.')

with open(summary_path, 'r', encoding='utf-8') as f:
    summary = json.load(f)

with open(experiments_path, 'r', encoding='utf-8') as f:
    experiments_history = json.load(f)

print(f'Experimentos carregados: {len(summary["experiments"])}')

## 1. Tabela comparativa: baseline vs otimizado

In [ ]:
rows = []
baseline = summary['baseline']

for exp in summary['experiments']:
    model_type = exp['model_type']
    b = baseline[model_type]
    o = exp['optimized_metrics']
    rows.append({
        'experiment': exp['name'],
        'model': 'Logistic Regression' if model_type == 'lr' else 'Random Forest',
        'baseline_recall': b['recall'],
        'optimized_recall': o['recall'],
        'delta_recall': round(o['recall'] - b['recall'], 4),
        'baseline_f1': b['f1'],
        'optimized_f1': o['f1'],
        'delta_f1': round(o['f1'] - b['f1'], 4),
        'runtime_seconds': exp.get('runtime_seconds'),
    })

df_compare = pd.DataFrame(rows).sort_values('experiment')
df_compare

## 2. Evolucao do fitness por geracao

In [ ]:
plt.figure(figsize=(10, 5))
for exp in experiments_history:
    name = exp.get('name', exp['model_type'].upper())
    history = exp['history']
    generations = [h['generation'] for h in history]
    best = [h['best_fitness'] for h in history]
    plt.plot(generations, best, label=f'{name} - best')

plt.xlabel('Geracao')
plt.ylabel('Fitness (0.6*Recall + 0.4*F1)')
plt.title('Curva de evolucao do fitness por experimento')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 3. Melhores hiperparametros encontrados

In [ ]:
params_rows = []
for exp in summary['experiments']:
    params_rows.append({
        'experiment': exp['name'],
        'model_type': exp['model_type'],
        'best_fitness': exp['ga']['best_fitness'],
        'stop_reason': exp['ga']['stop_reason'],
        'best_params': exp['ga']['best_params'],
    })

pd.DataFrame(params_rows)

## 4. Analise rapida

- O objetivo principal e melhorar **recall** com controle de F1.
- Compare os deltas de recall/f1 na tabela para decidir qual configuracao levar para producao/relatorio.
- Se necessario, rode uma segunda bateria com mais geracoes e CV maior para maior estabilidade.